Fixes applied:
- Background = 0 (standard)
- Foreground classes = 1–5
- Dataset size control
- Train/Val/Test split
- EDA included


# =====================
# CELL 1: INSTALL
# =====================
# !pip install segmentation-models-pytorch

# =====================
# CELL 2: IMPORTS
# =====================

In [1]:
!pip install segmentation-models-pytorch
!pip install albumentations   # for data augmentation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import cv2
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

import segmentation_models_pytorch as smp


# =====================
# CELL 3: CONFIG
# =====================

In [3]:
IMAGE_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/images"
ANNOT_DIR = "/kaggle/input/datasets/iharshsinha/deepfashion2-top5-processed/processed/train/annos"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256

NUM_CLASSES = 6  # 0 background + 5 classes
MAX_IMAGES = 2000  # 🔥 control dataset size here---not used now

label_map = {
    "short sleeve top": 1,
    "trousers": 2,
    "shorts": 3,
    "long sleeve top": 4,
    "skirt": 5
}

# Save label map
with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=4)


# =====================
# CELL 4: EDA
# =====================

In [4]:
def compute_class_distribution(annot_dir, max_files=None):
    counter = Counter()
    files = [f for f in os.listdir(annot_dir) if f.endswith(".json")]

    if max_files:
        files = files[:max_files]

    for file in files:
        with open(os.path.join(annot_dir, file)) as f:
            data = json.load(f)

        for k in data:
            if not k.startswith("item"):
                continue

            cat = data[k]["category_name"]
            if cat in label_map:
                counter[cat] += 1

    print("Class Distribution:")
    for k, v in counter.items():
        print(f"{k}: {v}")



# =====================
# CELL 5: MASK GENERATION
# =====================

In [5]:
def polygon_to_mask(polygons, class_id, mask):
    for poly in polygons:
        pts = np.array(poly).reshape(-1, 2).astype(np.int32)
        cv2.fillPoly(mask, [pts], class_id)


def generate_mask(json_path, image_shape):
    h, w = image_shape
    mask = np.zeros((h, w), dtype=np.uint8)  # background = 0

    with open(json_path) as f:
        data = json.load(f)

    for k in data:
        if not k.startswith("item"):
            continue

        item = data[k]
        cat = item["category_name"]

        if cat not in label_map:
            continue

        class_id = label_map[cat]
        polygons = item["segmentation"]

        polygon_to_mask(polygons, class_id, mask)

    return mask



# =====================

# CELL 6: DATASET (UPDATED WITH AUGMENTATION)
# =====================

In [6]:
import albumentations as A

class FashionDataset(Dataset):
    def __init__(self, image_dir, annot_dir, files, transform=None):
        self.image_dir = image_dir
        self.annot_dir = annot_dir
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]

        img_path = os.path.join(self.image_dir, img_name)
        json_path = os.path.join(self.annot_dir, img_name.replace(".jpg", ".json"))

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w, _ = image.shape
        mask = generate_mask(json_path, (h, w))

        # 🔥 Augmentation (train only)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        # Resize
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        # Normalize
        image = image / 255.0
        image = (image - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225] # new addition, as resnet-34 expects imagenet normalization something
        image = np.transpose(image, (2, 0, 1)).astype(np.float32)

        return torch.tensor(image), torch.tensor(mask).long()


# NEW CELL: DEFINE AUGMENTATIONS

In [7]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
])

# No augmentation for val/test
val_transform = None
test_transform = None




# =====================
# CELL 7: SPLIT DATA (UPDATED for augmentations)
# =====================


* Stage	Setting
* Debug sample_fraction=0.02
* Mid	sample_fraction=0.1
* Final	sample_fraction=0.3–0.4


In [8]:
import random
from torch.utils.data import Subset 

# 🔥 STEP 1: Create ONE shared file list
files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")]

random.seed(42)
random.shuffle(files)

# 🔥 Apply subsampling ONCE
sample_fraction = 0.3   # change this later (0.1, 0.3, etc.)
num_samples = int(len(files) * sample_fraction)
files = files[:num_samples]

print("Total samples used:", len(files))

# 🔥 STEP 2: Split indices
total = len(files)
train_size = int(0.8 * total)
val_size = int(0.1 * total)
test_size = total - train_size - val_size


# 🔥 set seed for torch RNG
seed = 42
g = torch.Generator()
g.manual_seed(seed)

# 🔥 deterministic split
train_indices, val_indices, test_indices = random_split(
    range(total),
    [train_size, val_size, test_size],
    generator=g
)


# 🔥 STEP 3: Create datasets (same files, different transforms)
train_ds_full = FashionDataset(IMAGE_DIR, ANNOT_DIR, files, transform=train_transform)
val_ds_full   = FashionDataset(IMAGE_DIR, ANNOT_DIR, files, transform=None)
# test_ds_full  = FashionDataset(IMAGE_DIR, ANNOT_DIR, files, transform=None)
test_ds_full = val_ds_full

# 🔥 STEP 4: Apply subsets
train_ds = Subset(train_ds_full, train_indices.indices)
val_ds   = Subset(val_ds_full, val_indices.indices)
test_ds  = Subset(test_ds_full, test_indices.indices)

# 🔥 STEP 5: DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=4
)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")


Total samples used: 43252
Train: 34601, Val: 4325, Test: 4326


In [9]:
# =====================
# SAVE DATA SPLIT (IMPORTANT)
# =====================

train_files = [files[i] for i in train_indices.indices]
val_files   = [files[i] for i in val_indices.indices]
test_files  = [files[i] for i in test_indices.indices]

split = {
    "train": train_files,
    "val": val_files,
    "test": test_files
}

with open("data_split.json", "w") as f:
    json.dump(split, f)

print("Data split saved to data_split.json")

Data split saved to data_split.json



# =====================
# CELL 8: MODEL
# =====================

In [10]:
def get_model(pretrained=True):
    return smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet" if pretrained else None,
        in_channels=3,
        classes=NUM_CLASSES
    )



# =====================
# CELL 9: LOSS   ** delete the class_weight file for now as it was ran on 0.01 sample
# =====================

In [11]:
# def compute_class_weights(dataset, num_classes=6):
#     counts = np.zeros(num_classes)

#     for i in range(len(dataset)):
#         _, mask = dataset[i]
#         mask = mask.numpy()

#         for cls in range(num_classes):
#             counts[cls] += np.sum(mask == cls)

#     # avoid division by zero
#     counts = np.maximum(counts, 1)

#     weights = 1.0 / counts
#     weights = weights / weights.sum() * num_classes

#     return torch.tensor(weights, dtype=torch.float32)


# # create clean train dataset (no augmentation)
# clean_train_ds = Subset(val_ds_full, train_indices.indices)

# class_weights = compute_class_weights(clean_train_ds).to(DEVICE)

# ce_loss = nn.CrossEntropyLoss(weight=class_weights)
# # dice_loss = smp.losses.DiceLoss(mode="multiclass", from_logits=True)
# dice_loss = smp.losses.DiceLoss(
#     mode="multiclass",
#     from_logits=True,
#     classes=[1,2,3,4,5]
# )# skipping background here!!!

# def loss_fn(pred, target):
#     return ce_loss(pred, target) + dice_loss(pred, target)


# =====================
# CELL 9: LOSS (FINAL FIXED VERSION)
# =====================

WEIGHTS_PATH = "class_weights_5k.pt"

def compute_class_weights(dataset, num_classes=6, max_samples=5000):
    counts = np.zeros(num_classes)

    # take first 5k samples (NO shuffling as per your request)
    indices = list(range(len(dataset)))[:min(max_samples, len(dataset))]

    for i in indices:
        _, mask = dataset[i]
        mask = mask.numpy()

        # fast counting
        bincount = np.bincount(mask.flatten(), minlength=num_classes)
        counts += bincount

    # avoid division by zero
    counts = np.maximum(counts, 1)

    # inverse frequency
    weights = 1.0 / counts

    # normalize
    weights = weights / weights.sum() * num_classes

    return torch.tensor(weights, dtype=torch.float32)


# =====================
# LOAD OR COMPUTE WEIGHTS
# =====================

# ✅ correct dataset: TRAIN (no augmentation)
clean_train_ds = Subset(val_ds_full, train_indices.indices)# val_ds_full is the full train_ds without augmentation

if os.path.exists(WEIGHTS_PATH):
    print("Loading saved class weights...")
    class_weights = torch.load(WEIGHTS_PATH)
else:
    print("Computing class weights (5k samples)...")
    class_weights = compute_class_weights(clean_train_ds)
    torch.save(class_weights, WEIGHTS_PATH)

# move to device
class_weights = class_weights.to(DEVICE)

print("Class weights:", class_weights)


# =====================
# LOSS FUNCTIONS
# =====================

ce_loss = nn.CrossEntropyLoss(weight=class_weights)

dice_loss = smp.losses.DiceLoss(
    mode="multiclass",
    from_logits=True,
    classes=[1, 2, 3, 4, 5]  # ignore background
)

def loss_fn(pred, target):
    return ce_loss(pred, target) + dice_loss(pred, target)


Computing class weights (5k samples)...
Class weights: tensor([0.0655, 0.4879, 1.1452, 1.7460, 0.8645, 1.6909], device='cuda:0')



# =====================
# CELL 10: METRICS
# =====================

In [12]:
def compute_iou(pred, target):
    ious = []

    for cls in range(1, NUM_CLASSES):
        pred_inds = (pred == cls)
        target_inds = (target == cls)

        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()

        if union == 0:
            continue

        ious.append(intersection / union)

    return np.mean(ious) if ious else 0


def compute_dice(pred, target):
    dices = []

    for cls in range(1, NUM_CLASSES):
        pred_inds = (pred == cls)
        target_inds = (target == cls)

        intersection = (pred_inds & target_inds).sum().item()
        total = pred_inds.sum().item() + target_inds.sum().item()

        if total == 0:
            continue

        dices.append((2 * intersection) / total)

    return np.mean(dices) if dices else 0



# =====================
# CELL 11: TRAIN LOOP
# =====================

In [13]:

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        out = model(imgs)
        loss = loss_fn(out, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_model(model, loader):
    model.eval()
    iou_total, dice_total = 0, 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

            out = model(imgs)
            preds = torch.argmax(out, dim=1)

            iou_total += compute_iou(preds, masks)
            dice_total += compute_dice(preds, masks)

    return iou_total / len(loader), dice_total / len(loader)



# =====================
# CELL 12: RUN TRAINING
# =====================

In [14]:

# new version with early stopping
def run_training(pretrained=True, epochs=30, save_path="model.pth", patience=5):
    model = get_model(pretrained)

    if torch.cuda.device_count() > 1:
        print("Using multiple GPUs")
        model = nn.DataParallel(model)

    model = model.to(DEVICE)

    lr = 1e-4 if pretrained else 1e-3
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_iou = 0
    counter = 0  # 🔥 early stopping counter

    for epoch in range(epochs):
        loss = train_epoch(model, train_loader, optimizer)
        val_iou, val_dice = eval_model(model, val_loader)

        print(f"Epoch {epoch}: Loss={loss:.4f}, IoU={val_iou:.4f}, Dice={val_dice:.4f}")

        # 🔥 Check improvement
        if val_iou > best_iou:
            best_iou = val_iou
            counter = 0  # reset counter

            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), save_path)
            else:
                torch.save(model.state_dict(), save_path)

            print(f"Saved best model at epoch {epoch}")

        else:
            counter += 1
            print(f"No improvement. Early stop counter: {counter}/{patience}")

        # 🔥 Early stopping condition
        if counter >= patience:
            print("Early stopping triggered")
            break

    return model


# =====================
# CELL 13: EXECUTE
# =====================

In [15]:
# compute_class_distribution(ANNOT_DIR)    no point in going throught the entire thing

print("\nTraining from scratch")
model_scratch = run_training(pretrained=False,epochs=35, save_path="unet_scratch.pth")

print("\nTraining with transfer learning")
model_tl = run_training(pretrained=True, epochs=35, save_path="unet_tl.pth")


Training from scratch
Using multiple GPUs
Epoch 0: Loss=2.3139, IoU=0.1240, Dice=0.1962
Saved best model at epoch 0
Epoch 1: Loss=1.9360, IoU=0.2386, Dice=0.3428
Saved best model at epoch 1
Epoch 2: Loss=1.7379, IoU=0.2882, Dice=0.4032
Saved best model at epoch 2
Epoch 3: Loss=1.5923, IoU=0.3052, Dice=0.4229
Saved best model at epoch 3
Epoch 4: Loss=1.4938, IoU=0.3296, Dice=0.4483
Saved best model at epoch 4
Epoch 5: Loss=1.4124, IoU=0.3774, Dice=0.4990
Saved best model at epoch 5
Epoch 6: Loss=1.3445, IoU=0.3773, Dice=0.4986
No improvement. Early stop counter: 1/5
Epoch 7: Loss=1.2793, IoU=0.3871, Dice=0.5087
Saved best model at epoch 7
Epoch 8: Loss=1.2377, IoU=0.4056, Dice=0.5263
Saved best model at epoch 8
Epoch 9: Loss=1.1815, IoU=0.4114, Dice=0.5311
Saved best model at epoch 9
Epoch 10: Loss=1.1343, IoU=0.4313, Dice=0.5540
Saved best model at epoch 10
Epoch 11: Loss=1.0936, IoU=0.4519, Dice=0.5717
Saved best model at epoch 11
Epoch 12: Loss=1.0536, IoU=0.4430, Dice=0.5631
No imp

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Using multiple GPUs
Epoch 0: Loss=1.5058, IoU=0.4203, Dice=0.5441
Saved best model at epoch 0
Epoch 1: Loss=1.1591, IoU=0.4752, Dice=0.5936
Saved best model at epoch 1
Epoch 2: Loss=1.0419, IoU=0.4715, Dice=0.5893
No improvement. Early stop counter: 1/5
Epoch 3: Loss=0.9587, IoU=0.5032, Dice=0.6187
Saved best model at epoch 3
Epoch 4: Loss=0.8976, IoU=0.4986, Dice=0.6132
No improvement. Early stop counter: 1/5
Epoch 5: Loss=0.8513, IoU=0.5129, Dice=0.6260
Saved best model at epoch 5
Epoch 6: Loss=0.8024, IoU=0.5494, Dice=0.6600
Saved best model at epoch 6
Epoch 7: Loss=0.7655, IoU=0.5424, Dice=0.6535
No improvement. Early stop counter: 1/5
Epoch 8: Loss=0.7338, IoU=0.5421, Dice=0.6537
No improvement. Early stop counter: 2/5
Epoch 9: Loss=0.6961, IoU=0.5394, Dice=0.6502
No improvement. Early stop counter: 3/5
Epoch 10: Loss=0.6709, IoU=0.5514, Dice=0.6611
Saved best model at epoch 10
Epoch 11: Loss=0.6472, IoU=0.5696, Dice=0.6774
Saved best model at epoch 11
Epoch 12: Loss=0.6300, IoU=0